# Week 17 — Company Filing Extractor (Structured Outputs)

## Introduction

This notebook extracts key fields (revenue, net profit, period, basis, guidance, risks) from
NSE/BSE quarterly results filings and returns them as validated JSON. It compares four approaches:

1. **Prompt-only JSON**: ask for JSON in the prompt and parse it
2. **JSON mode**: the API guarantees valid JSON, not the schema
3. **Schema-constrained output**: the API forces output to match a JSON schema
4. **Tool calling**: the model fills a function schema

Goal: how accurately can an LLM extract these fields, and how much do validation,
retry, and few-shot examples improve reliability?

In [5]:
!pip install groq pdfplumber

  Using cached groq-1.7.0-py3-none-any.whl.metadata (16 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 35.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 72.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 121.1 MB/s  0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [pdfplumber]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [6]:
import pdfplumber
from pydantic import BaseModel,Field,ConfigDict,ValidationError,field_validator
from typing import Optional,Literal
from groq import Groq
from pathlib import Path
import json,hashlib,time,re
import pandas as pd

In [7]:
# Local storage for caching
CACHE_DIR = Path("/content/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

## Dataset

**Source:** NSE/BSE corporate announcements (quarterly results). 34 filings:
10 dev (used for prompt tuning) and 24 test (used only for final evaluation).
Only the first 5 pages of each PDF are used, because results tables usually appear early.
This is a limitation: a table on a later page would be missed.

In [8]:
docs_dir = Path("/content/documents")
docs=[]
for file_path in docs_dir.glob("*.pdf"):
  full_text = []
  with pdfplumber.open(file_path) as pdf:
    for page in pdf.pages[:5]:
      text = page.extract_text()
      if text:
        full_text.append(text)

      '''tables = page.extract_tables()
      if tables:
        for table in tables:
          for row in table:
            cleaned_row = [str(cell).strip() if cell else "" for cell in row]
            full_text.append("|".join(cleaned_row)+"\n")
      ''' # the extract_text method is already reading the table so there's no need of extract_tables separately

  docs.append({
      "file_name":file_path.stem,
      "content":"".join(full_text)
  })


### Dev / Test Split

Prompts and few-shot examples were tuned on the dev set only. The test set was run
once for final numbers.

In [9]:
dev = [ doc for doc in docs[:10]]
test = [doc for doc in docs[10:]]

In [13]:
from google.colab import userdata
key = userdata.get('GROQ_API_KEY')
client = Groq(api_key = key)
MODEL = "openai/gpt-oss-120b"

## Labeling Guide

**Written before labeling. If I hit a new edge case, I update this cell and re-check earlier labels.**

- **revenue_cr** = Revenue from operations (not total income). For banks/NBFCs: *decide your rule, e.g. total income or interest earned + other income, and write it here.*
- **net_profit_cr** = Profit after tax. Negative for a loss. For banks: *write your rule here.*
- **basis**: if both consolidated and standalone are given, use consolidated and set `basis = "consolidated"`.
- **Units**: all amounts in ₹ crore, rounded to 2 decimals. 1 crore = 10 million = 100 lakh.
- **Null rule**: if a value is not printed in the document, label it `null`. Never calculate or infer it (for example, don't compute YoY growth yourself).
- **risks**: only risks explicitly mentioned in the document, maximum 5.
- **Evidence**: for revenue and profit, copy the exact sentence or table row that shows the figure.

## Extraction Methods

### Method A: Prompt-only JSON
The prompt asks for JSON and nothing else. No API-level guarantee.

In [7]:
def method_a(doc_name,context):

  messages = []
  messages.append({
      "role":"system",
      "content":'''

      RESPOND ONLY in valid JSON OBJECT form, do not include any other text.
      Analyse the text inside <document></document> and return the following data in JSON notation:

      {company_name:<Company name as written in the document>,
      period:<Reporting period, e.g. 'Q2 FY27'. null if not stated>,
      basis:<Which results are extracted. null if unclear>,
      revenue_cr:<Total revenue from operations in INR crore. Convert lakh/million to crore. null if not stated>,
      net_profit_cr:<Net profit after tax in INR crore. Negative for a loss. null if not stated>,
      revenue_yoy_pct:<Year-on-year revenue growth in percent, only if stated in the document. null otherwise>,
      guidance:<Management outlook or guidance in one sentence. null if none given>,
      risks:<Up to 5 risks explicitly mentioned. Empty list if none>,
      revenue_evidence:<Exact sentence or table row from the document that shows the revenue figure>,
      profit_evidence:<Exact sentence or table row from the document that shows the profit figure>
      }
  '''
  })
  messages.append({
      "role":"user",
      "content":f"<document>{context}</document>"
  })

  key = hashlib.md5(json.dumps([MODEL, messages], sort_keys=True).encode()).hexdigest()[:8]
  path = CACHE_DIR / f"{doc_name}__A__{key}.json"
  if path.exists():
    return json.loads(path.read_text())["raw"]

  resp = client.chat.completions.create(
      model=MODEL,temperature=0.0,
      messages=messages
  )
  text = resp.choices[0].message.content
  path.write_text(json.dumps({"raw": text}))
  return text


**Method A result:** 10/10 parsed. Typical failure: `basis` outside the allowed values.

## Method 2: JSON Mode

In [8]:
def method_b(doc_name,context):

  messages = []
  messages.append({
      "role":"system",
      "content":'''

      RESPOND ONLY in valid JSON OBJECT form, do not include any other text.
      Analyse the text inside <document></document> and return the following data in JSON notation:

      {company_name:<Company name as written in the document>,
      period:<Reporting period, e.g. 'Q2 FY27'. null if not stated>,
      basis:<Which results are extracted. null if unclear>,
      revenue_cr:<Total revenue from operations in INR crore. Convert lakh/million to crore. null if not stated>,
      net_profit_cr:<Net profit after tax in INR crore. Negative for a loss. null if not stated>,
      revenue_yoy_pct:<Year-on-year revenue growth in percent, only if stated in the document. null otherwise>,
      guidance:<Management outlook or guidance in one sentence. null if none given>,
      risks:<Up to 5 risks explicitly mentioned. Empty list if none>,
      revenue_evidence:<Exact sentence or table row from the document that shows the revenue figure>,
      profit_evidence:<Exact sentence or table row from the document that shows the profit figure>

      }
  '''
  })
  messages.append({
      "role":"user",
      "content":f"<document>{context}</document>"
  })

  key = hashlib.md5(json.dumps([MODEL, messages], sort_keys=True).encode()).hexdigest()[:8]
  path = CACHE_DIR / f"{doc_name}__B__{key}.json"
  if path.exists():
    return json.loads(path.read_text())["raw"]

  resp = client.chat.completions.create(
      model=MODEL,temperature=0.0,
      response_format={"type":"json_object"},
      messages=messages
  )
  text = resp.choices[0].message.content
  path.write_text(json.dumps({"raw": text}))
  return text


## Method 3: Structured Output
## Schema

Fields are `Optional` on purpose: if a value isn't in the document the answer is `null`,
not a guess. Validators run after the types check: period format, non-negative revenue.
They allow `null`, so a missing value doesn't crash the pipeline.

In [15]:
class FilingExtract(BaseModel):
  model_config = ConfigDict(extra="forbid") #required by Groq

  company_name :str= Field(...,description="Company name as written in the document")
  period: Optional[str] = Field(..., description="Reporting period, e.g. 'Q2 FY27'. null if not stated")
  basis: Optional[Literal["consolidated", "standalone","Consolidated", "Standalone"]] = Field(..., description="Which results are extracted. null if unclear")
  revenue_cr: Optional[float] = Field(..., description='''
  Total revenue from operations in INR crore. Convert lakh/million to crore. null if not stated"
  IMPORTANT:
    - Check the table heading for units (INR, thousands, lakhs, millions, crores).
    - Convert the original figure into INR crore.
    - Do not confuse total income with revenue from operations.
    - Extract the correct reporting period column.
    - Return null if the figure cannot be identified confidently.
  ''')
  net_profit_cr: Optional[float] = Field(..., description="Net profit after tax in INR crore. Negative for a loss. null if not stated")
  revenue_yoy_pct: Optional[float] = Field(..., description="Year-on-year revenue growth in percent, only if stated in the document. null otherwise")
  guidance: Optional[str] = Field(..., description="Management outlook or guidance in one sentence. null if none given")
  risks: list[str] = Field(..., description="Up to 5 risks explicitly mentioned. Empty list if none")
  revenue_evidence: Optional[str] = Field(..., description="Exact sentence or table row from the document that shows the revenue figure")
  profit_evidence: Optional[str] = Field(..., description="Exact sentence or table row from the document that shows the profit figure")

  @field_validator("revenue_cr")
  @classmethod
  def validate_revenue(cls,revenue:float):
    if revenue is None or revenue<0.0:
      raise ValueError('Revenue Cannot be negative')
    return revenue

  @field_validator("period")
  @classmethod
  def validate_period(cls,period:str):

    if period is None: return None
    pattern = r'^(?:Q[1-4]|H[1-2])\sFY\d{2}$'
    cleaned_value = period.strip()

    if not re.match(pattern, cleaned_value):
      raise ValueError(
                "Period must be formatted exactly like 'Q1 FY27' (Q1-Q4) or 'H1 FY27' (H1-H2)."
            )
    return cleaned_value

print(FilingExtract.model_json_schema())


{'additionalProperties': False, 'properties': {'company_name': {'description': 'Company name as written in the document', 'title': 'Company Name', 'type': 'string'}, 'period': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'description': "Reporting period, e.g. 'Q2 FY27'. null if not stated", 'title': 'Period'}, 'basis': {'anyOf': [{'enum': ['consolidated', 'standalone', 'Consolidated', 'Standalone'], 'type': 'string'}, {'type': 'null'}], 'description': 'Which results are extracted. null if unclear', 'title': 'Basis'}, 'revenue_cr': {'anyOf': [{'type': 'number'}, {'type': 'null'}], 'description': '\n  Total revenue from operations in INR crore. Convert lakh/million to crore. null if not stated"\n  IMPORTANT:\n    - Check the table heading for units (INR, thousands, lakhs, millions, crores).\n    - Convert the original figure into INR crore.\n    - Do not confuse total income with revenue from operations.\n    - Extract the correct reporting period column.\n    - Return null if the f

In [44]:
def method_c(doc_name,context):

  messages = []
  messages.append({
      "role":"system",
      "content":'''

      RESPOND ONLY in valid JSON OBJECT form, do not include any other text.
      Analyse the text inside <document></document> and return the data in JSON notation
  '''
  })
  messages.append({
      "role":"user",
      "content":f"<document>{context}</document>"
  })

  key = hashlib.md5(json.dumps([MODEL, messages], sort_keys=True).encode()).hexdigest()[:8]
  path = CACHE_DIR / f"{doc_name}__C__{key}.json"
  if path.exists():
    return json.loads(path.read_text())["raw"]

  schema = FilingExtract.model_json_schema()
  schema["required"] = list(schema.get("properties", {}).keys())
  resp = client.chat.completions.create(
      model=MODEL,temperature=0.0,
      messages=messages,
      response_format={
          "type":"json_schema",
          "json_schema":{
              "name":"FilingExtract",
              "strict":True,
              "schema":schema
          }
      }
  )
  text = resp.choices[0].message.content
  FilingExtract.model_validate_json(text)
  path.write_text(json.dumps({"raw": text}))
  return text


## Method 4: Tool Calling

In [45]:
def method_d(doc_name,context):
  messages = []
  messages.append({
      "role":"system",
      "content":'''
      Analyse the financial document and extract the required fields.
            You MUST call the record_filing tool.
            If a field is not available, return null where permitted.
            Do not invent financial figures.
  '''
  })
  messages.append({
      "role":"user",
      "content":f"<document>{context}</document>"
  })

  key = hashlib.md5(json.dumps([MODEL, messages], sort_keys=True).encode()).hexdigest()[:8]
  path = CACHE_DIR / f"{doc_name}__D__{key}.json"
  if path.exists():
    return FilingExtract.model_validate(json.loads(path.read_text())["raw"]
        ).model_dump_json()


  schema = FilingExtract.model_json_schema()
  schema["required"] = list(schema.get("properties", {}).keys())
  tools = [{
    "type": "function",
    "function": {
        "name": "record_filing",
        "description": "Record the extracted fields from a company results announcement",
        "parameters": schema,
        "strict":True
    },
}]
  resp = client.chat.completions.create(
      model=MODEL,temperature=0.0,
      messages=messages,
      tools=tools,
      tool_choice={
          "type":"function",
          "function":{
              "name":"record_filing"
          }
      }
  )
  tool_call = resp.choices[0].message.tool_calls[0]
  args = json.loads(tool_call.function.arguments)
  text = FilingExtract.model_validate(args)
  output = text.model_dump()
  path.write_text(json.dumps({"raw": output}))
  return json.dumps(output)

In [48]:
resp =[]
methods = [method_a,method_b,method_c,method_d]
for doc in dev:
  print(f"File Name: {doc["file_name"]}")
  for idx,method in enumerate(methods):
    try:
      print(f"Method {chr(ord('A')+idx)}")
      text = method(doc["file_name"],doc['content'])
      print(text)
      resp.append(json.loads(text))
    except Exception as e:
      print(f"Unable to JSONify : {e}")

    time.sleep(5)

File Name: AGI Infra
Method A
{"company_name":"AGI INFRA LIMITED","period":"Q2 FY27","basis":"Unaudited","revenue_cr":100.1884,"net_profit_cr":30.6586,"revenue_yoy_pct":null,"guidance":null,"risks":[],"revenue_evidence":"Revenue from operations 10,018.84","profit_evidence":"Net Profit Loss for the period from continuing operations 3,065.86"}
Method B
{"company_name":"AGI INFRA LIMITED","period":"Second quarter FY27","basis":"Unaudited","revenue_cr":100.1884,"net_profit_cr":30.6586,"revenue_yoy_pct":null,"guidance":null,"risks":[],"revenue_evidence":"Revenue from operations 10,018.84","profit_evidence":"Net Profit Loss for the period from continuing operations 3,065.86"}
Method C
{"company_name":"AGI INFRA LIMITED","period":"Q2 FY27","basis":"Standalone","revenue_cr":100.1884,"net_profit_cr":30.6586,"revenue_yoy_pct":null,"guidance":null,"risks":[],"revenue_evidence":"Revenue from operations 10,018.84","profit_evidence":"Net Profit Loss for the period from continuing operations 3,065.86

| Method | Valid on first try | Typical failure |
|-|-|-|
| A: prompt-only JSON | 10/10 | `basis` not in allowed values |
| B: JSON mode | 10/10 | `basis` not in allowed values |
| C: schema-constrained | 10/10 | none |
| D: tool calling | 10/10 | unit conversion to crore |

Valid JSON does not mean correct values: Method D validated revenue and net profit but they were not accurate as unit was not converted to Crores

In [54]:
df = pd.read_csv("/content/maniifest.csv")
df


,company_name,period,basis,revenue_cr,net_profit_cr,revenue_yoy_pct,guidance,risks,revenue_evidence,profit_evidence
0,AGI,Q2 2027,Standalone,100.1884,30.6586,NaN,NaN,NaN,"Revenue from operations 10,018.84",Net Profit Loss for the period from continuing...
1,ELIXIR CAPITAL LIMITED,Second quarter,Standalone,0.7486,0.7204,NaN,NaN,NaN,Revenue from operations 74.68,Net Profit Loss for the period from continuing...
2,WEST COAST PAPER MILLS LIMITED,Second quarter,Standalone,812.9714,110.5092,NaN,NaN,NaN,"Revenue from operations 81,297.14",Net Profit Loss for the period from continuing...
3,Renaissance Global Limited,Second quarter,Standalone,423.8543,8.8725,NaN,NaN,NaN,"Revenue from operations 42,385.43",Net Profit Loss for the period from continuing...
4,The Victoria Mills Limited,Second quarter,Standalone,19.5000,3.3949,NaN,NaN,NaN,"Revenue from operations 1,950.00",Net Profit Loss for the period from continuing...
5,TREJHARA SOLUTIONS LIMITED,Second quarter,Standalone,33.6000,0.2183,NaN,NaN,NaN,"Revenue from operations 3,360.16",Net Profit Loss for the period from continuing...
6,R R SECURITIES LIMITED,Second quarter,Standalone,0.0000,-0.0257,NaN,NaN,NaN,Revenue from operations 0.00,Net Profit Loss for the period from continuing...
7,GINI SILK,Second quarter,Standalone,11.4805,0.8634,NaN,NaN,NaN,"Revenue from operations 1,148.05",Net Profit Loss for the period from continuing...
8,INNOVATORS FAÃ‡ADE SYSTEMS LIMITED,Half Yearly,Standalone,137.3411,8.6900,NaN,NaN,NaN,"Revenue from operations 13,734.11",Net Profit Loss for the period from continuing...
9,VEDIC AYURVEDA,Second quarter,Standalone,0.9498,0.1124,NaN,NaN,NaN,Revenue from operations 94.98,Net Profit Loss for the period from continuing...


## Validation(Method 2) and retry  with few shot prompting

In [11]:

def build_messages(doc_text):
  messages = []
  messages.append({
      "role":"system",
      "content":'''

      RESPOND ONLY in valid JSON OBJECT form, do not include any other text.
      Analyse the text inside <document></document> and return the following data in JSON notation:

      {company_name:<Company name as written in the document>,
      period:<Reporting period, e.g. 'Q2 FY27'. null if not stated>,
      basis:<Which results are extracted. null if unclear>,
      revenue_cr:<Total revenue from operations in INR crore. Convert lakh/million to crore. null if not stated>,
      net_profit_cr:<Net profit after tax in INR crore. Negative for a loss. null if not stated>,
      revenue_yoy_pct:<Year-on-year revenue growth in percent, only if stated in the document. null otherwise>,
      guidance:<Management outlook or guidance in one sentence. null if none given>,
      risks:<Up to 5 risks explicitly mentioned. Empty list if none>,
      revenue_evidence:<Exact sentence or table row from the document that shows the revenue figure>,
      profit_evidence:<Exact sentence or table row from the document that shows the profit figure>

      Example 1:
      'company_name': 'Sudarshan Chemical Industries Limited',
      'period': 'Q2 FY27',
      'basis': 'Standalone',
      'revenue_cr': 777.2,
      'net_profit_cr': 61.0,
      'revenue_yoy_pct': None,
      'guidance': None,
      'risks': [],
      'revenue_evidence':
      'Revenue from operations 777.20 1,454.60',
      'profit_evidence': 'Net profit (loss) for the period from continuing operations 61.00 118.80'

      Example 2:
      'company_name': 'R R SECURITIES LIMITED',
      'period': 'Q2 FY27',
      'basis': 'Standalone',
      'revenue_cr': 0.0,
      'net_profit_cr': -0.0257,
      'revenue_yoy_pct': None,
      'guidance': None,
      'risks': [],
      'revenue_evidence': 'Revenue from operations 0.00 0.00',
      'profit_evidence': 'Net Profit Loss for the period from continuing operations (2.57) 6.26'

      }
  '''
  })
  messages.append({
      "role":"user",
      "content":f"<document>{doc_text}</document>"
  })
  return messages

def call_llm(messages):
  resp = client.chat.completions.create(
      model=MODEL, temperature=0,
      messages=messages,response_format={"type":"json_object"}
  )

  return resp.choices[0].message.content
def extract_with_retry(doc_text, max_retries=2):
    messages = build_messages(doc_text)
    for attempt in range(max_retries + 1):
        raw = call_llm(messages)
        try:
            FilingExtract.model_validate_json(raw)
            return json.loads(raw),attempt
        except ValidationError as e:
            messages += [
                {"role": "assistant", "content": raw},
                {"role": "user", "content": f"Your output failed validation:\n{e}\nReturn corrected JSON only."},
            ]
    return None, max_retries + 1

In [16]:
for doc in dev:
  print(f"File name: {doc['file_name']}")
  _,tries = extract_with_retry(doc['content'])
  print(_)
  print(f"Retries it took : {tries}")

File name: Honasa Consumer Limited
{'company_name': 'Honasa Consumer Limited', 'period': 'Q2 FY27', 'basis': 'Standalone', 'revenue_cr': 689.4, 'net_profit_cr': 71.701, 'revenue_yoy_pct': None, 'guidance': None, 'risks': [], 'revenue_evidence': 'Revenue from operations 6,894.00 13,857.18', 'profit_evidence': 'Net Profit Loss for the period from continuing operations 717.01 1,560.13'}
Retries it took : 0
File name: SUNIL INDUSTRIES LIMITED
{'company_name': 'SUNIL INDUSTRIES LIMITED', 'period': 'Q2 FY27', 'basis': 'Standalone', 'revenue_cr': 106.1345, 'net_profit_cr': 2.7567, 'revenue_yoy_pct': None, 'guidance': None, 'risks': [], 'revenue_evidence': 'Revenue from operations 10,613.45 17,658.94', 'profit_evidence': 'Net Profit Loss for the period from continuing operations 275.67 437.41'}
Retries it took : 0
File name: GINI SILK
{'company_name': 'GINI SILK MILLS LIMITED', 'period': 'Q2 FY27', 'basis': 'Standalone', 'revenue_cr': 11.48, 'net_profit_cr': 0.86, 'revenue_yoy_pct': None, 'gui

## Validation and Retry

On a validation failure the model gets its own output plus the Pydantic error message
and tries again (maximum 2 retries).

| Model | Prompt | First-try validity | Final validity | Retry rate | Evidence match |
|-|-|-|-|-|-|
| gpt-oss-120b | zero-shot | 2/10 | 10/10 | 80% | 100% |
| qwen | few-shot | 10/10 | 10/10 | 0% | 100% |
| gpt-oss-120b | few-shot | 7/10| 10/10 | 30% | 100% |

In [17]:
for doc in test:
  print(f"File name: {doc['file_name']}")
  _,tries = extract_with_retry(doc['content'])
  print(_)
  print(f"Retries it took : {tries}")

File name: Danube
{'company_name': 'Danube Industries Ltd', 'period': 'Q2 FY27', 'basis': 'Standalone', 'revenue_cr': 27.1974, 'net_profit_cr': 0.6738, 'revenue_yoy_pct': None, 'guidance': None, 'risks': [], 'revenue_evidence': 'Revenue from operations 2,719.74 4,452.45', 'profit_evidence': 'Net Profit Loss for the period from continuing operations 67.38 92.27'}
Retries it took : 0
File name: PIX TRANSMISSIONS LTD.
{'company_name': 'PIX TRANSMISSIONS LTD.', 'period': 'Q2 FY27', 'basis': 'Standalone', 'revenue_cr': 148.9087, 'net_profit_cr': 29.5297, 'revenue_yoy_pct': None, 'guidance': None, 'risks': [], 'revenue_evidence': 'Revenue from operations 14,890.87 28,344.08', 'profit_evidence': 'Net Profit Loss for the period from continuing operations 2,952.97 7,781.83'}
Retries it took : 0
File name: Indo Amines Limited
{'company_name': 'Indo Amines Limited', 'period': 'Q2 FY27', 'basis': 'Standalone', 'revenue_cr': 380.3018, 'net_profit_cr': 6.8299, 'revenue_yoy_pct': None, 'guidance': No

## Evaluations

|Metric	|Value|
|-|-|
|Documents evaluated (test set)	|24|
|Schema validity: first try	|100%(qwen) & 91.67%(gpt-120b)|
|Schema validity: after retry	|100% (for both)|
|Retry rate	|0%(qwen model) & 8.33%(gpt-120b)|
|Evidence-match rate|	97.917%|
|Field accuracy: revenue_cr	|95.83%|
|Field accuracy: net_profit_cr|	100%|


## Insights

The most reliable method on the dev set was Method 2 using validation Loop. Valid JSON was not the same as correct
valuesas seen using tool calling . Few-shot examples changed first-try validity from 20% to 100%. Retries fixed
8 of 8 failures, but cannot fix a wrong value that is valid JSON. The most common error was unit conversion error.